<a href="https://colab.research.google.com/github/Shabd45261/CastLynk/blob/master/Qwen3_TTS_Voice_Cloning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys

# Install dependencies with FlashAttention 2 for faster inference
print('Updating setuptools and wheel...')
!{sys.executable} -m pip install -U setuptools wheel

print('Installing ninja and packaging...')
!{sys.executable} -m pip install ninja packaging

print('Installing FlashAttention 2...')
!MAX_JOBS=4 {sys.executable} -m pip install -U flash-attn --no-build-isolation

# Install TTS and audio packages
print('Installing qwen-tts, soundfile, and gradio...')
!{sys.executable} -m pip install -q qwen-tts soundfile gradio
!apt-get install -qq sox libsox-fmt-all

# Verify FlashAttention installation
try:
    import flash_attn
    print("✅ FlashAttention 2 successfully imported!")
except ImportError:
    print("❌ FlashAttention 2 import failed. Please check the installation.")


print("✅ All dependencies installed successfully!")
print("   - FlashAttention 2 (faster generation)")
print("   - Qwen-TTS package")
print("   - SoundFile (audio processing)")
print("   - Gradio (web interface)")

Updating setuptools and wheel...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 21.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
Installing ninja and packaging...
Installing FlashAttention 2...
  Using cached flash_attn-2.8.3.post1.tar.gz (8.5 MB)
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.9 MB/s eta 0:00:00


In [2]:
import torch
import soundfile as sf
import gradio as gr
import numpy as np
from qwen_tts import Qwen3TTSModel

print("Loading Qwen3-TTS Base model with FlashAttention 2...")
print("(This takes 3-5 minutes on first run)")

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",  # Enable FlashAttention 2
)

print("✅ Model loaded successfully with FlashAttention 2!")
print("   You can now clone voices in the interface below.")



********
********
 
Loading Qwen3-TTS Base model with FlashAttention 2...
(This takes 3-5 minutes on first run)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

ImportError: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package flash_attn seems to be not installed. Please refer to the documentation of https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2 to install Flash Attention 2.

In [3]:
def clone_voice(new_text, language, reference_audio, ref_transcript):
    """
    Clone a voice and generate speech in that cloned voice.

    Args:
        new_text: The text you want the cloned voice to speak
        language: Target language for synthesis
        reference_audio: Audio file path (3-10 seconds of clear speech)
        ref_transcript: Exact transcript of what's said in reference audio
    """
    try:
        if reference_audio is None:
            return None, "❌ Please upload a reference audio file (3-10 seconds recommended)"

        if not ref_transcript or ref_transcript.strip() == "":
            return None, "❌ Please provide the transcript of your reference audio"

        # Generate voice clone using official Qwen3-TTS API
        wavs, sr = model.generate_voice_clone(
            text=new_text,
            language=language,
            ref_audio=reference_audio,
            ref_text=ref_transcript,
        )

        # Handle output format
        if isinstance(wavs, (list, tuple)):
            audio_data = np.array(wavs[0])
        else:
            audio_data = np.array(wavs)

        return (int(sr), audio_data), f"✅ Voice cloned successfully! Sample rate: {sr}Hz"

    except Exception as e:
        import traceback
        return None, f"❌ Error: {str(e)}\n\n{traceback.format_exc()}"

# Supported languages (10 major languages)
languages = [
    "Chinese", "English", "Japanese", "Korean",
    "German", "French", "Russian", "Portuguese",
    "Spanish", "Italian"
]

# Create Gradio interface
interface = gr.Interface(
    fn=clone_voice,
    inputs=[
        gr.Textbox(
            label="📝 NEW Text (what you want the cloned voice to say)",
            placeholder="Enter the text you want to speak in the cloned voice...",
            lines=4,
            value="Hello! This is my cloned voice speaking new words."
        ),
        gr.Dropdown(
            choices=languages,
            value="English",
            label="🌐 Language"
        ),
        gr.Audio(
            label="🎤 Reference Audio (3-10 seconds of clear speech)",
            type="filepath",
            sources=["upload", "microphone"]
        ),
        gr.Textbox(
            label="📄 Reference Audio Transcript",
            placeholder="Type EXACTLY what is spoken in the reference audio above...",
            lines=3,
            value=""
        )
    ],
    outputs=[
        gr.Audio(label="🔊 Cloned Voice Output", type="numpy"),
        gr.Textbox(label="📊 Status", lines=2)
    ],
    title="🎙️ Qwen3-TTS Voice Cloning",
    description="""
    **Clone any voice from just 3 seconds of audio!**

    **How to use:**
    1. Upload a 3-10 second audio clip of the voice you want to clone
    2. Type exactly what is said in that audio (the transcript)
    3. Enter the new text you want this voice to speak
    4. Click Submit and wait for your cloned voice!

    **Tips for best results:**
    - Use clear audio without background noise
    - Make sure your transcript matches the audio exactly
    - Longer reference audio (10-20 seconds) gives better results
    - Works in 10 languages!
    """,
    examples=[
        [
            "Welcome to my AI voice cloning demo!",
            "English",
            None,
            ""
        ],
        [
            "This technology is amazing!",
            "English",
            None,
            ""
        ]
    ]
)

# Launch the interface
print("🚀 Launching Gradio interface...")
interface.launch(share=True, debug=False)


🚀 Launching Gradio interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://722ba26ddab6c726df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
